In [ ]:
pip install pandas pyarrow fastparquet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd

# Read parquet file
df = pd.read_parquet("/content/cases.parquet")

# Save as CSV
df.to_csv("/content/case.csv", index=False)

print("Conversion done ✔")


Conversion done ✔


In [ ]:
import pandas as pd
import ast


In [ ]:
captions_df = pd.read_csv("/content/captions_and_labels.csv")
cases_df = pd.read_csv("/content/case.csv")
metadata_df = pd.read_csv("/content/metadata.csv")
abstracts_df = pd.read_csv("/content/abstracts.csv")
case_images_df = pd.read_csv("/content/case_images.csv")


In [ ]:
type(cases_df.loc[0, "cases"])


str

In [ ]:
print(cases_df.loc[0, "cases"][:200])


[{'age': 53.0, 'case_id': 'PMC3738355_01', 'case_text': 'A 53-year-old woman presented with a 10-year history of intermittent abdominal pain, swelling and continuous vomiting. The patient denied prese


In [ ]:
cases_df.shape

(72581, 2)

In [ ]:
import ast

def parse_cases(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return None
    return x


In [ ]:
cases_df["cases_parsed"] = cases_df["cases"].apply(parse_cases)


In [ ]:
cases_clean = cases_df.copy()


In [ ]:
cases_clean.shape


(72581, 3)

In [ ]:
cases_exploded = cases_clean.explode("cases_parsed").reset_index(drop=True)


In [ ]:
cases_flat = pd.json_normalize(cases_exploded["cases_parsed"])


In [ ]:
cases_flat["article_id"] = cases_exploded["article_id"].values


In [ ]:
cases_flat = cases_flat.dropna(subset=["case_id", "case_text"])


In [ ]:
cases_flat.head()


,age,case_id,case_text,gender,article_id
0,53.0,PMC3738355_01,A 53-year-old woman presented with a 10-year h...,Female,PMC3738355
1,69.0,PMC5015624_01,A 69-year-old Caucasian female with coronary a...,Female,PMC5015624
2,60.0,PMC6381877_01,A 60-year-old male smoker presented with persi...,Male,PMC6381877
4,30.0,PMC5287946_01,A 30-year-old man came to Peking Union Medical...,Male,PMC5287946
5,40.0,PMC9106225_01,A 40-year-old female patient was referred for ...,Female,PMC9106225


In [ ]:
images_core = captions_df[[
    "patient_id",
    "file",
    "caption",
    "image_component",
    "case_substring",
    "image_type",
    "image_subtype",
    "radiology_region",
    "radiology_view",
    "ml_labels_for_supervised_classification",
    "gt_labels_for_semisupervised_classification"
]].rename(columns={"patient_id": "case_id"})


In [ ]:
import ast

def parse_list(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return x
    return x

for col in [
    "case_substring",
    "ml_labels_for_supervised_classification",
    "gt_labels_for_semisupervised_classification"
]:
    images_core[col] = images_core[col].apply(parse_list)


In [ ]:
images_agg = images_core.groupby("case_id").agg({
    "file": list,
    "caption": list,
    "image_component": list,
    "case_substring": list,
    "image_type": list,
    "image_subtype": list,
    "radiology_region": list,
    "radiology_view": list,
    "ml_labels_for_supervised_classification": list,
    "gt_labels_for_semisupervised_classification": list
}).reset_index()


In [ ]:
images_agg = images_agg.rename(columns={
    "file": "image_files",
    "caption": "image_captions",
    "image_component": "image_components",
    "case_substring": "case_substrings",
    "image_type": "image_types",
    "image_subtype": "image_subtypes",
    "radiology_region": "radiology_regions",
    "radiology_view": "radiology_views",
    "ml_labels_for_supervised_classification": "ml_labels",
    "gt_labels_for_semisupervised_classification": "semi_labels"
})


In [ ]:
final_df = cases_flat.merge(
    images_agg,
    on="case_id",
    how="left"
)


In [ ]:
final_df.isna().sum()


,0
age,1397
case_id,0
case_text,0
gender,0
article_id,0
image_files,35253
image_captions,35253
image_components,35253
case_substrings,35253
image_types,35253


In [ ]:
final_df = final_df.drop(
    columns=[
        "image_files",
        "image_captions",
        "image_types",
        "image_subtypes",
        "radiology_regions"
    ],
    errors="ignore"
)


In [ ]:
final_df = final_df.merge(images_agg, on="case_id", how="left")


In [ ]:
final_df = cases_flat.merge(images_agg, on="case_id", how="left")


In [ ]:
final_df.head()

,age,case_id,case_text,gender,article_id,image_files,image_captions,image_components,case_substrings,image_types,image_subtypes,radiology_regions,radiology_views,ml_labels,semi_labels
0,53.0,PMC3738355_01,A 53-year-old woman presented with a 10-year h...,Female,PMC3738355,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,69.0,PMC5015624_01,A 69-year-old Caucasian female with coronary a...,Female,PMC5015624,"[PMC5015624_OC-05-14-g-001_undivided_1_1.webp,...",[Slit lamp biomicroscopy of the anterior segme...,"[undivided, undivided]","[[Figure 1, Fig. 1], [Figure 4, Fig. 4]]","[endoscopy, radiology]","[airway_endoscopy, mri]","[nan, head]","[nan, sagittal]","[[endoscopy, airway_endoscopy], [head, radiolo...","[[], [radiology, mri, angiography, mra]]"
2,60.0,PMC6381877_01,A 60-year-old male smoker presented with persi...,Male,PMC6381877,"[PMC6381877_cro-0012-0091-g01_a_1_2.webp, PMC6...",[Chest X-ray and chest CT images before initia...,"[a, b, a, b]","[[Fig. 1a, b], [Fig. 1a, b], [Fig. 2a, b], [Fi...","[radiology, radiology, radiology, radiology]","[x_ray, ct, x_ray, ct]","[thorax, thorax, thorax, thorax]","[frontal, axial, frontal, axial]","[[thorax, radiology, frontal, x_ray], [thorax,...","[[radiology], [radiology], [radiology], [radio..."
3,30.0,PMC5287946_01,A 30-year-old man came to Peking Union Medical...,Male,PMC5287946,"[PMC5287946_medi-96-e5657-g002_A_1_12.webp, PM...",[Histopathological and immunohistochemical exa...,"[a, b, c, d, e, f, g, h, i, j, k, l]","[[Fig. 2A-K], [Fig. 2A-K], [Fig. 2A-K], [Fig. ...","[pathology, pathology, pathology, pathology, p...","[h&e, h&e, immunostaining, immunostaining, imm...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[[pathology, h&e], [pathology, h&e], [patholog...","[[pathology], [pathology], [pathology, immunos..."
4,40.0,PMC9106225_01,A 40-year-old female patient was referred for ...,Female,PMC9106225,[PMC9106225_JOMFP-26-101-g001_undivided_1_1.we...,"[Clinical image shows a well-defined, roughly ...","[undivided, undivided, undivided, undivided]","[[Figure 1], [Figure 2], [Figure 3, Figures 3]...","[pathology, pathology, pathology, pathology]","[h&e, h&e, pas, alcian_blue]","[nan, nan, nan, nan]","[nan, nan, nan, nan]","[[pathology, h&e], [pathology, h&e], [patholog...","[[], [pathology, h&e], [pathology, pas], [path..."


In [ ]:
final_df.shape

(61437, 15)

In [ ]:
def clean_numpy_arrays(s):
    if not isinstance(s, str):
        return None

    # array(  -> [
    s = re.sub(r"array\(", "[", s)

    # , dtype=object) -> ]
    s = re.sub(r",\s*dtype=object\)", "]", s)

    return s


In [ ]:
import pandas as pd
import numpy as np
import ast
import re


In [ ]:
case_images_df["case_images_clean"] = (
    case_images_df["case_images"].apply(clean_numpy_arrays)
)


In [ ]:
def safe_parse(s):
    try:
        return ast.literal_eval(s)
    except Exception:
        return None


In [ ]:
case_images_df["case_images_parsed"] = (
    case_images_df["case_images_clean"].apply(safe_parse)
)


In [ ]:
idx = case_images_df["case_images_parsed"].first_valid_index()
type(case_images_df.loc[idx, "case_images_parsed"])


list

In [ ]:
import numpy as np

rows = []

for _, row in case_images_df.iterrows():
    parsed = row["case_images_parsed"]
    article_id = row["article_id"]

    if not isinstance(parsed, list):
        continue

    for case_block in parsed:
        case_id = case_block.get("case_id")
        image_list = case_block.get("case_image_list")

        # Convert numpy array → list
        if isinstance(image_list, np.ndarray):
            image_list = image_list.tolist()

        if not isinstance(image_list, list):
            continue

        # 🔑 FIX: flatten if nested list
        if len(image_list) == 1 and isinstance(image_list[0], list):
            image_list = image_list[0]

        for img in image_list:
            if not isinstance(img, dict):
                continue

            rows.append({
                "case_id": case_id,
                "article_id": article_id,
                "file": img.get("file"),
                "caption": img.get("caption"),
                "image_id": img.get("image_id"),
                "tag": img.get("tag"),
                "text_references": (
                    img.get("text_references").tolist()
                    if isinstance(img.get("text_references"), np.ndarray)
                    else img.get("text_references")
                )
            })


In [ ]:
image_level = pd.DataFrame(rows)


In [ ]:
print(image_level.shape)
image_level.head()


(155029, 7)


,case_id,article_id,file,caption,image_id,tag,text_references
0,PMC3738355_01,PMC3738355,10.1258_arsr.2012.120031-fig1.jpg,(a) Axial unenhanced CT image demonstrates an ...,PMC3738355_01_10.1258_arsr.2012.120031-fig1.jpg,ARSR-12-0031F1,[[Axial unenhanced CT scan (Fig. 1a) confirmed...
1,PMC3738355_01,PMC3738355,10.1258_arsr.2012.120031-fig2.jpg,(a) Variably sized vascular channels lined wit...,PMC3738355_01_10.1258_arsr.2012.120031-fig2.jpg,ARSR-12-0031F2,[[Histologic examination (Fig. 2) revealed tha...
2,PMC5015624_01,PMC5015624,OC-05-14-g-001.jpg,Slit lamp biomicroscopy of the anterior segmen...,PMC5015624_01_OC-05-14-g-001.jpg,F1,[[Slit lamp biomicroscopy showed iris neovascu...
3,PMC5015624_01,PMC5015624,OC-05-14-g-002.jpg,Optical coherence tomography of the right eye ...,PMC5015624_01_OC-05-14-g-002.jpg,F2,[[Optical coherence tomography showed no evide...
4,PMC5015624_01,PMC5015624,OC-05-14-g-003.jpg,Fluorescein angiogram showing significantly de...,PMC5015624_01_OC-05-14-g-003.jpg,F3,[[Fluorescein angiography exhibited delayed ar...


In [ ]:
case_images_agg = image_level.groupby("case_id").agg({
    "file": list,
    "caption": list,
    "image_id": list,
    "tag": list,
    "text_references": list
}).reset_index()


In [ ]:
final_df = final_df.merge(case_images_agg, on="case_id", how="left")


In [ ]:
final_df.head()

,age,case_id,case_text,gender,article_id,image_files,image_captions,image_components,case_substrings,image_types,image_subtypes,radiology_regions,radiology_views,ml_labels,semi_labels,file,caption,image_id,tag,text_references
0,53.0,PMC3738355_01,A 53-year-old woman presented with a 10-year h...,Female,PMC3738355,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[10.1258_arsr.2012.120031-fig1.jpg, 10.1258_ar...",[(a) Axial unenhanced CT image demonstrates an...,[PMC3738355_01_10.1258_arsr.2012.120031-fig1.j...,"[ARSR-12-0031F1, ARSR-12-0031F2]",[[[Axial unenhanced CT scan (Fig. 1a) confirme...
1,69.0,PMC5015624_01,A 69-year-old Caucasian female with coronary a...,Female,PMC5015624,"[PMC5015624_OC-05-14-g-001_undivided_1_1.webp,...",[Slit lamp biomicroscopy of the anterior segme...,"[undivided, undivided]","[[Figure 1, Fig. 1], [Figure 4, Fig. 4]]","[endoscopy, radiology]","[airway_endoscopy, mri]","[nan, head]","[nan, sagittal]","[[endoscopy, airway_endoscopy], [head, radiolo...","[[], [radiology, mri, angiography, mra]]","[OC-05-14-g-001.jpg, OC-05-14-g-002.jpg, OC-05...",[Slit lamp biomicroscopy of the anterior segme...,"[PMC5015624_01_OC-05-14-g-001.jpg, PMC5015624_...","[F1, F2, F3, F4]",[[[Slit lamp biomicroscopy showed iris neovasc...
2,60.0,PMC6381877_01,A 60-year-old male smoker presented with persi...,Male,PMC6381877,"[PMC6381877_cro-0012-0091-g01_a_1_2.webp, PMC6...",[Chest X-ray and chest CT images before initia...,"[a, b, a, b]","[[Fig. 1a, b], [Fig. 1a, b], [Fig. 2a, b], [Fi...","[radiology, radiology, radiology, radiology]","[x_ray, ct, x_ray, ct]","[thorax, thorax, thorax, thorax]","[frontal, axial, frontal, axial]","[[thorax, radiology, frontal, x_ray], [thorax,...","[[radiology], [radiology], [radiology], [radio...","[cro-0012-0091-g01.jpg, cro-0012-0091-g02.jpg,...",[Chest X-ray and chest CT images before initia...,"[PMC6381877_01_cro-0012-0091-g01.jpg, PMC63818...","[F1, F2, F3]","[[[In the imaging test of full body, chest X-r..."
3,30.0,PMC5287946_01,A 30-year-old man came to Peking Union Medical...,Male,PMC5287946,"[PMC5287946_medi-96-e5657-g002_A_1_12.webp, PM...",[Histopathological and immunohistochemical exa...,"[a, b, c, d, e, f, g, h, i, j, k, l]","[[Fig. 2A-K], [Fig. 2A-K], [Fig. 2A-K], [Fig. ...","[pathology, pathology, pathology, pathology, p...","[h&e, h&e, immunostaining, immunostaining, imm...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[[pathology, h&e], [pathology, h&e], [patholog...","[[pathology], [pathology], [pathology, immunos...","[medi-96-e5657-g001.jpg, medi-96-e5657-g002.jpg]",[MRI for the abnormalities in the sellar regio...,"[PMC5287946_01_medi-96-e5657-g001.jpg, PMC5287...","[F1, F2]",[[[The magnetic resonance imaging (MRI) demons...
4,40.0,PMC9106225_01,A 40-year-old female patient was referred for ...,Female,PMC9106225,[PMC9106225_JOMFP-26-101-g001_undivided_1_1.we...,"[Clinical image shows a well-defined, roughly ...","[undivided, undivided, undivided, undivided]","[[Figure 1], [Figure 2], [Figure 3, Figures 3]...","[pathology, pathology, pathology, pathology]","[h&e, h&e, pas, alcian_blue]","[nan, nan, nan, nan]","[nan, nan, nan, nan]","[[pathology, h&e], [pathology, h&e], [patholog...","[[], [pathology, h&e], [pathology, pas], [path...","[JOMFP-26-101-g001.jpg, JOMFP-26-101-g002.jpg,...","[Clinical image shows a well-defined, roughly ...","[PMC9106225_01_JOMFP-26-101-g001.jpg, PMC91062...","[F1, F2, F3, F4]",[[[Clinical examination revealed a well-define...


In [ ]:
final_df.shape

(61437, 20)

In [ ]:
metadata_df.columns
# ['article_metadata', 'article_id']


Index(['article_metadata', 'article_id'], dtype='object')

In [ ]:
def clean_numpy_arrays(s):
    if not isinstance(s, str):
        return None

    s = re.sub(r"array\(", "[", s)
    s = re.sub(r",\s*dtype=object\)", "]", s)

    return s


In [ ]:
metadata_df["metadata_clean"] = (
    metadata_df["article_metadata"].apply(clean_numpy_arrays)
)


In [ ]:
def safe_parse(s):
    try:
        return ast.literal_eval(s)
    except:
        return None


In [ ]:
metadata_df["metadata_parsed"] = (
    metadata_df["metadata_clean"].apply(safe_parse)
)


In [ ]:
idx = metadata_df["metadata_parsed"].first_valid_index()
type(metadata_df.loc[idx, "metadata_parsed"])


dict

In [ ]:
metadata_flat = pd.json_normalize(
    metadata_df["metadata_parsed"]
)


In [ ]:
metadata_flat["article_id"] = metadata_df["article_id"].values


In [ ]:
metadata_flat.columns


Index(['authors', 'case_amount', 'doi', 'journal', 'journal_detail',
       'keywords', 'license', 'link', 'major_mesh_terms', 'mesh_terms',
       'pmcid', 'pmid', 'title', 'year', 'article_id'],
      dtype='object')

In [ ]:
def to_list(x):
    if isinstance(x, np.ndarray):
        return x.tolist()
    return x


In [ ]:
for col in ["authors", "keywords", "mesh_terms", "major_mesh_terms"]:
    if col in metadata_flat.columns:
        metadata_flat[col] = metadata_flat[col].apply(to_list)


In [ ]:
final_df = final_df.merge(
    metadata_flat[
        [
            "article_id",
            "title",
            "journal",
            "year",
            "doi",
            "keywords",
            "mesh_terms",
            "major_mesh_terms",
            "authors",
            "license"
        ]
    ],
    on="article_id",
    how="left"
)


In [ ]:
final_df.loc[
    final_df["article_id"] == "PMC3738355",
    ["title", "journal", "year", "keywords"]
].head(1)



,title,journal,year,keywords
0,Littoral cell angioma mimicking hepatic tumor,Acta Radiol Short Rep,2012,"[[vascular tumor, diagnostic imaging, liver, p..."


In [ ]:
final_df.head()

,age,case_id,case_text,gender,article_id,image_files,image_captions,image_components,case_substrings,image_types,...,text_references,title,journal,year,doi,keywords,mesh_terms,major_mesh_terms,authors,license
0,53.0,PMC3738355_01,A 53-year-old woman presented with a 10-year h...,Female,PMC3738355,NaN,NaN,NaN,NaN,NaN,...,[[[Axial unenhanced CT scan (Fig. 1a) confirme...,Littoral cell angioma mimicking hepatic tumor,Acta Radiol Short Rep,2012,10.1258/arsr.2012.120031,"[[vascular tumor, diagnostic imaging, liver, p...",[[Case Reports]],[[]],"[[Wenhua Liang, Jingjing Lu, Mingwei Qin, Xint...",CC BY-NC
1,69.0,PMC5015624_01,A 69-year-old Caucasian female with coronary a...,Female,PMC5015624,"[PMC5015624_OC-05-14-g-001_undivided_1_1.webp,...",[Slit lamp biomicroscopy of the anterior segme...,"[undivided, undivided]","[[Figure 1, Fig. 1], [Figure 4, Fig. 4]]","[endoscopy, radiology]",...,[[[Slit lamp biomicroscopy showed iris neovasc...,Central retinal artery occlusion following las...,GMS Ophthalmol Cases,2015,10.3205/oc000036,"[[mri, aortic arch, carotid artery stenosis, c...",[[Case Reports]],[[]],"[[Payal J Shah, Brian Ellis, Lauren R DiGiovin...",CC BY
2,60.0,PMC6381877_01,A 60-year-old male smoker presented with persi...,Male,PMC6381877,"[PMC6381877_cro-0012-0091-g01_a_1_2.webp, PMC6...",[Chest X-ray and chest CT images before initia...,"[a, b, a, b]","[[Fig. 1a, b], [Fig. 1a, b], [Fig. 2a, b], [Fi...","[radiology, radiology, radiology, radiology]",...,"[[[In the imaging test of full body, chest X-r...",Promising Combination Therapy with Bevacizumab...,Case Rep Oncol,2019,10.1159/000493088,"[[bevacizumab, egfr mutation, erlotinib, lung ...",[[Case Reports]],[[]],"[[Nobuhiko Seki, Maika Natsume, Ryosuke Ochiai...",CC BY-NC
3,30.0,PMC5287946_01,A 30-year-old man came to Peking Union Medical...,Male,PMC5287946,"[PMC5287946_medi-96-e5657-g002_A_1_12.webp, PM...",[Histopathological and immunohistochemical exa...,"[a, b, c, d, e, f, g, h, i, j, k, l]","[[Fig. 2A-K], [Fig. 2A-K], [Fig. 2A-K], [Fig. ...","[pathology, pathology, pathology, pathology, p...",...,[[[The magnetic resonance imaging (MRI) demons...,Malignant adenohypophysis spindle cell oncocyt...,Medicine (Baltimore),2017,10.1097/MD.0000000000005657,None,"[[Adenoma, Oxyphilic / pathology, Ki-67 Antige...","[[Adenoma, Oxyphilic / pathology, Ki-67 Antige...","[[Xiangyi Kong, Dongmei Li, Yanguo Kong, Dingr...",CC BY
4,40.0,PMC9106225_01,A 40-year-old female patient was referred for ...,Female,PMC9106225,[PMC9106225_JOMFP-26-101-g001_undivided_1_1.we...,"[Clinical image shows a well-defined, roughly ...","[undivided, undivided, undivided, undivided]","[[Figure 1], [Figure 2], [Figure 3, Figures 3]...","[pathology, pathology, pathology, pathology]",...,[[[Clinical examination revealed a well-define...,A novel mucocele: Myxoglobulosis,J Oral Maxillofac Pathol,2022,10.4103/jomfp.jomfp_214_21,"[[mucocele, mucus extravasation cyst, myxoglob...",[[Case Reports]],[[]],"[[Rutuja Gajanan Vidhale, Subraj Shetty, Nikit...",CC BY-NC-SA


In [ ]:
abstracts_df["abstract"] = (
    abstracts_df["abstract"]
    .astype(str)
    .str.replace("\n", " ")
    .str.strip()
)


In [ ]:
final_df = final_df.merge(
    abstracts_df,
    on="article_id",
    how="left"
)


In [ ]:
final_df.loc[
    final_df["article_id"] == "PMC3738355",
    ["title", "abstract"]
].head(1)


,title,abstract
0,Littoral cell angioma mimicking hepatic tumor,Littoral cell angioma is a rare vascular tumor...


In [ ]:
final_df.head()

,age,case_id,case_text,gender,article_id,image_files,image_captions,image_components,case_substrings,image_types,...,title,journal,year,doi,keywords,mesh_terms,major_mesh_terms,authors,license,abstract
0,53.0,PMC3738355_01,A 53-year-old woman presented with a 10-year h...,Female,PMC3738355,NaN,NaN,NaN,NaN,NaN,...,Littoral cell angioma mimicking hepatic tumor,Acta Radiol Short Rep,2012,10.1258/arsr.2012.120031,"[[vascular tumor, diagnostic imaging, liver, p...",[[Case Reports]],[[]],"[[Wenhua Liang, Jingjing Lu, Mingwei Qin, Xint...",CC BY-NC,Littoral cell angioma is a rare vascular tumor...
1,69.0,PMC5015624_01,A 69-year-old Caucasian female with coronary a...,Female,PMC5015624,"[PMC5015624_OC-05-14-g-001_undivided_1_1.webp,...",[Slit lamp biomicroscopy of the anterior segme...,"[undivided, undivided]","[[Figure 1, Fig. 1], [Figure 4, Fig. 4]]","[endoscopy, radiology]",...,Central retinal artery occlusion following las...,GMS Ophthalmol Cases,2015,10.3205/oc000036,"[[mri, aortic arch, carotid artery stenosis, c...",[[Case Reports]],[[]],"[[Payal J Shah, Brian Ellis, Lauren R DiGiovin...",CC BY,Objective: Ocular ischemic syndrome is a rare ...
2,60.0,PMC6381877_01,A 60-year-old male smoker presented with persi...,Male,PMC6381877,"[PMC6381877_cro-0012-0091-g01_a_1_2.webp, PMC6...",[Chest X-ray and chest CT images before initia...,"[a, b, a, b]","[[Fig. 1a, b], [Fig. 1a, b], [Fig. 2a, b], [Fi...","[radiology, radiology, radiology, radiology]",...,Promising Combination Therapy with Bevacizumab...,Case Rep Oncol,2019,10.1159/000493088,"[[bevacizumab, egfr mutation, erlotinib, lung ...",[[Case Reports]],[[]],"[[Nobuhiko Seki, Maika Natsume, Ryosuke Ochiai...",CC BY-NC,"In lung cancer, several potential mechanisms o..."
3,30.0,PMC5287946_01,A 30-year-old man came to Peking Union Medical...,Male,PMC5287946,"[PMC5287946_medi-96-e5657-g002_A_1_12.webp, PM...",[Histopathological and immunohistochemical exa...,"[a, b, c, d, e, f, g, h, i, j, k, l]","[[Fig. 2A-K], [Fig. 2A-K], [Fig. 2A-K], [Fig. ...","[pathology, pathology, pathology, pathology, p...",...,Malignant adenohypophysis spindle cell oncocyt...,Medicine (Baltimore),2017,10.1097/MD.0000000000005657,None,"[[Adenoma, Oxyphilic / pathology, Ki-67 Antige...","[[Adenoma, Oxyphilic / pathology, Ki-67 Antige...","[[Xiangyi Kong, Dongmei Li, Yanguo Kong, Dingr...",CC BY,Adenohypophysis spindle cell oncocytoma (ASCO)...
4,40.0,PMC9106225_01,A 40-year-old female patient was referred for ...,Female,PMC9106225,[PMC9106225_JOMFP-26-101-g001_undivided_1_1.we...,"[Clinical image shows a well-defined, roughly ...","[undivided, undivided, undivided, undivided]","[[Figure 1], [Figure 2], [Figure 3, Figures 3]...","[pathology, pathology, pathology, pathology]",...,A novel mucocele: Myxoglobulosis,J Oral Maxillofac Pathol,2022,10.4103/jomfp.jomfp_214_21,"[[mucocele, mucus extravasation cyst, myxoglob...",[[Case Reports]],[[]],"[[Rutuja Gajanan Vidhale, Subraj Shetty, Nikit...",CC BY-NC-SA,Oral extravasation mucoceles are among the mos...


In [ ]:
final_df.to_csv("multimodal_cases.csv", index=False)


In [ ]:
final_df.shape

(61437, 30)

In [ ]:
final_df["has_text"] = final_df["case_text"].notna()
final_df["has_images"] = final_df["image_files"].notna()

final_df.groupby(["has_text", "has_images"]).size()


has_text  has_images
True      False         35253
          True          26184
dtype: int64